# Team Classification

Per-game step: classify every detected player as Team A, Team B, or Other (ref/GK/staff).

## Workflow

**One-time labeling per game** (this notebook):
1. Set `GAME_SLUG` and run **Pass 1** → detects players across 6 evenly-spaced windows (3 FH + 3 SH), broadcast frames only
2. Label each crop in the widget: **A** (Team A) · **B** (Team B) · **X** (Other: ref/GK/staff)
3. Run **Pass 2** → fits KNN (k=5) classifier, saves two files to `output/classifiers/`:
   - `{slug}_classifier.pkl` — fitted PCA + KNN model
   - `{slug}_labels.npz` — raw labeled embeddings + labels (source of truth for future refitting)

**At event-detection time** (`03_event_detection.ipynb`):
- Load the saved classifier — no re-labeling needed
- Apply in real-time on the full match

## Pipeline
1. **Detect + Track** — YOLOv8m + BoT-SORT across 6 × 90s windows spread over both halves
2. **Broadcast filter** — skip replays/close-ups using scoreboard edge density
3. **Embed** — YOLOv8n-seg masks player silhouettes; frozen ResNet18 → 512-dim vectors
4. **PCA** — 512 → 32 dims
5. **Label** — assign ~100 crops per team in the widget
6. **Supervised fit** — KNN (k=5) classifier in PCA space
7. **Save** — model pkl + labels npz serialized to `output/classifiers/`

In [1]:
import sys
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import importlib
import io
from pathlib import Path
from tqdm import tqdm
from PIL import Image as PILImage
import ipywidgets as widgets
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import src.config, src.video_utils, src.detection, src.team_classifier, src.tracking, src.visualization, src.segmentation
for mod in [src.config, src.video_utils, src.detection, src.team_classifier, src.tracking, src.visualization, src.segmentation]:
    importlib.reload(mod)

from src.config import Config
from src.video_utils import open_video, get_video_info, create_video_writer
from src.detection import PlayerDetector, BallInterpolator
from src.team_classifier import TeamClassifier
from src.tracking import Tracker
from src.visualization import Annotator

print('Loaded.')

Loaded.


In [ ]:
import joblib

def _edge_density(frame, roi):
    """Canny edge density in an ROI — used to detect broadcast scoreboard overlay."""
    x1, y1, x2, y2 = roi
    crop = frame[y1:y2, x1:x2]
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    return cv2.Canny(gray, 50, 150).mean()

def _iou(a, b):
    """Intersection-over-union for two [x1,y1,x2,y2] boxes."""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    if inter == 0:
        return 0.0
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union

def run_pass1(slug, video_path, fh_start, sh_start, total_frames, fps,
              n_windows_per_half=3, window_sec=90, embed_sample_n=15):
    """
    Run Pass 1 for one game: detect, track, collect embeddings + best crops.
    Returns (track_embeddings, track_crops).
    track_crops[tid] = (crop_bgr, area, is_clean)
    """
    fh_end = sh_start - 1
    sh_end = total_frames - 1

    # Calibrate broadcast threshold from frames near kickoff
    cap = cv2.VideoCapture(str(video_path))
    cal_densities = []
    for fi in range(fh_start, min(fh_start + 300, total_frames), 30):
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame = cap.read()
        if ret:
            cal_densities.append(_edge_density(frame, Config.SCOREBOARD_ROI))
    sb_threshold = np.percentile(cal_densities, 30) * 0.5 if cal_densities else 1.0
    print(f'  Broadcast threshold: {sb_threshold:.3f}')

    # Evenly-spaced window starts across both halves
    win_starts = []
    for i in range(n_windows_per_half):
        win_starts.append(int(fh_start + (i + 0.5) * (fh_end - fh_start) / n_windows_per_half))
    for i in range(n_windows_per_half):
        win_starts.append(int(sh_start + (i + 0.5) * (sh_end - sh_start) / n_windows_per_half))
    win_len = int(fps * window_sec)

    game_clf         = TeamClassifier()
    track_embeddings = {}
    track_crops      = {}

    for win_idx, win_start in enumerate(win_starts):
        win_end     = min(win_start + win_len, total_frames - 1)
        clip_frames = win_end - win_start
        tid_offset  = win_idx * 10_000
        win_detector = PlayerDetector()
        n_sampled = n_skipped = 0

        cap.set(cv2.CAP_PROP_POS_FRAMES, win_start)
        half_label = 'FH' if win_idx < n_windows_per_half else 'SH'
        for i in tqdm(range(clip_frames),
                      desc=f'  {slug} win {win_idx+1}/{len(win_starts)} ({half_label})',
                      leave=False):
            ret, frame = cap.read()
            if not ret:
                break
            if _edge_density(frame, Config.SCOREBOARD_ROI) < sb_threshold:
                n_skipped += 1
                continue
            dets = win_detector.detect_with_tracking(frame)
            if i % embed_sample_n == 0 and dets['players']:
                all_bboxes = [p['bbox'] for p in dets['players']]
                embs = game_clf.collect_embeddings(frame, dets['players'])
                for p, emb in zip(dets['players'], embs):
                    raw_tid = p.get('track_id', -1)
                    if raw_tid < 0:
                        continue
                    tid = raw_tid + tid_offset
                    track_embeddings.setdefault(tid, []).append(emb)
                    bbox = p['bbox']
                    x1, y1, x2, y2 = map(int, bbox)
                    area = (x2 - x1) * (y2 - y1)
                    crop = frame[y1:y2, x1:x2].copy()
                    if crop.size == 0:
                        continue
                    others = [b for b in all_bboxes if b is not bbox]
                    is_clean = all(_iou(bbox, ob) < 0.1 for ob in others)
                    prev = track_crops.get(tid)
                    if (prev is None
                            or (is_clean and not prev[2])
                            or (is_clean == prev[2] and area > prev[1])):
                        track_crops[tid] = (crop, area, is_clean)
                n_sampled += 1

        print(f'  Win {win_idx+1} ({half_label}): {n_sampled} sampled, '
              f'{n_skipped} non-broadcast skipped → {len(track_embeddings)} tracks')

    cap.release()
    return track_embeddings, track_crops

## Per-Game Supervised Labeling

Workflow:
1. Set `GAME_SLUG` and run **Pass 1** → detects players across 6 evenly-spaced windows (3 FH + 3 SH), filters to broadcast wide-angle frames only
2. Label each crop in the widget: **A** (Team A) · **B** (Team B) · **X** (Other: ref/GK/staff)
3. Run **Pass 2** → fits KNN (k=5) classifier on your labels, classifies all tracks, saves model + raw labels to `output/classifiers/`

> The classifier propagates labels from your labeled tracks to all other tracks via KNN (k=5) in PCA space. The raw labeled embeddings are saved to `{slug}_labels.npz` so the classifier can be refit (e.g. different K) without re-labeling.

In [ ]:
# Labeling progress across all 16 games
clf_dir   = Config.OUTPUT_CLASSIFIERS_DIR
all_slugs = list(Config.MATCH_VIDEOS.keys())
labeled   = [s for s in all_slugs if (clf_dir / f'{s}_classifier.pkl').exists()]
pending   = [s for s in all_slugs if s not in labeled]

print(f'Progress: {len(labeled)}/{len(all_slugs)} games fully labeled\n')
for s in all_slugs:
    has_clf    = (clf_dir / f'{s}_classifier.pkl').exists()
    has_labels = (clf_dir / f'{s}_labels.npz').exists()
    has_pass1  = (clf_dir / f'{s}_pass1.pkl').exists()
    if has_clf and has_labels:
        tag = 'DONE'
    elif has_clf:
        tag = 'PKL '   # old format: no labels file
    elif has_pass1:
        tag = 'P1  '   # pass 1 saved, ready for labeling
    else:
        tag = '    '
    print(f'  [{tag}]  {s}')

if pending:
    print(f'\nNext up: {pending[0]}')

## Batch Pass 1 — All Games (Run Overnight)

Runs Pass 1 on every game that doesn't yet have a classifier or saved Pass 1 results.
Saves `{slug}_pass1.pkl` to `output/classifiers/` for each game.

Tomorrow: set `GAME_SLUG` below, run **Load Pass 1 results**, then do labeling + Pass 2.

In [ ]:
# Batch Pass 1 — skips games that already have a classifier or a saved pass1 file.
periods_data = json.loads((PROJECT_ROOT / 'data' / 'period_detection_results.json').read_text())
period_map   = {r['slug']: r for r in periods_data}

to_process = [
    s for s in Config.MATCH_VIDEOS
    if not (Config.OUTPUT_CLASSIFIERS_DIR / f'{s}_classifier.pkl').exists()
    and not (Config.OUTPUT_CLASSIFIERS_DIR / f'{s}_pass1.pkl').exists()
    and Path(str(Config.MATCH_VIDEOS[s])).exists()
    and s in period_map
]
print(f'Queued: {len(to_process)} games — {", ".join(to_process) or "none"}\n')

for slug in to_process:
    video_path   = Path(str(Config.MATCH_VIDEOS[slug]))
    period       = period_map[slug]
    fh_start     = period['first_half_start_frame']
    sh_start     = period['second_half_start_frame']
    cap_tmp      = cv2.VideoCapture(str(video_path))
    fps_g        = cap_tmp.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap_tmp.get(cv2.CAP_PROP_FRAME_COUNT))
    cap_tmp.release()

    print(f'\n{"="*60}\n{slug}\n{"="*60}')
    track_embeddings, track_crops = run_pass1(
        slug, video_path, fh_start, sh_start, total_frames, fps_g
    )

    save_path = Config.OUTPUT_CLASSIFIERS_DIR / f'{slug}_pass1.pkl'
    joblib.dump({'track_embeddings': track_embeddings, 'track_crops': track_crops}, save_path)
    sorted_tids = sorted(track_embeddings, key=lambda t: len(track_embeddings[t]), reverse=True)
    top_tids    = [t for t in sorted_tids if t in track_crops][:200]
    n_clean     = sum(1 for t in top_tids if track_crops[t][2])
    print(f'  Saved {save_path.name} — {len(track_embeddings)} tracks, '
          f'top {len(top_tids)} for labeling ({n_clean} clean)\n')

print('Done.')

In [38]:
GAME_SLUG          = 'bok-jed-2'   # <- change this per game
N_WINDOWS_PER_HALF = 3           # evenly-spaced windows per half (total = 2 × this)
WINDOW_SEC         = 90          # seconds per window
EMBED_SAMPLE_N     = 15          # sample embeddings every N frames
N_LABEL_CROPS      = 200         # crops to show in the labeling widget

video_path   = Path(str(Config.MATCH_VIDEOS[GAME_SLUG]))
periods_data = json.loads((PROJECT_ROOT / 'data' / 'period_detection_results.json').read_text())
period       = next(r for r in periods_data if r['slug'] == GAME_SLUG)
fh_start     = period['first_half_start_frame']
sh_start     = period['second_half_start_frame']
fh_end       = sh_start - 1      # period JSON has no first_half_end_frame

clf_path = Config.OUTPUT_CLASSIFIERS_DIR / f'{GAME_SLUG}_classifier.pkl'
if clf_path.exists():
    print(f'NOTE: {GAME_SLUG} already has a saved classifier at {clf_path}')
    print('Re-run to overwrite, or skip to the next game.')
else:
    print(f'Game: {GAME_SLUG} — {video_path.name}')
print(f'First half:  frame {fh_start} → {fh_end}')
print(f'Second half: frame {sh_start} → end')

Game: bok-jed-2 — BOKELJ-JEDINSTVO 1.CFL 4.KOLO 24.08.2025 CIJELA.mp4
First half:  frame 3266 → 74744
Second half: frame 74745 → end


In [ ]:
# Pass 1 (single game) — run this OR the Load cell below, not both.
cap_tmp      = cv2.VideoCapture(str(video_path))
fps          = cap_tmp.get(cv2.CAP_PROP_FPS)
total_frames = int(cap_tmp.get(cv2.CAP_PROP_FRAME_COUNT))
cap_tmp.release()

print(f'{GAME_SLUG} — {video_path.name}')
track_embeddings, track_crops = run_pass1(
    GAME_SLUG, video_path, fh_start, sh_start, total_frames, fps,
    n_windows_per_half=N_WINDOWS_PER_HALF, window_sec=WINDOW_SEC, embed_sample_n=EMBED_SAMPLE_N,
)
game_clf = TeamClassifier()   # fresh instance ready for Pass 2

save_path = Config.OUTPUT_CLASSIFIERS_DIR / f'{GAME_SLUG}_pass1.pkl'
joblib.dump({'track_embeddings': track_embeddings, 'track_crops': track_crops}, save_path)

sorted_tids = sorted(track_embeddings, key=lambda t: len(track_embeddings[t]), reverse=True)
sample_tids = [t for t in sorted_tids if t in track_crops][:N_LABEL_CROPS]
n_clean     = sum(1 for t in sample_tids if track_crops[t][2])
print(f'\n{len(track_embeddings)} total tracks → showing top {len(sample_tids)} for labeling'
      f'  ({n_clean} clean, {len(sample_tids) - n_clean} with player overlap)')

In [ ]:
# Load saved Pass 1 results — run this instead of the cell above if batch ran overnight.
pass1_path = Config.OUTPUT_CLASSIFIERS_DIR / f'{GAME_SLUG}_pass1.pkl'
if not pass1_path.exists():
    print(f'No saved Pass 1 for {GAME_SLUG}. Run Pass 1 cell above first.')
else:
    data             = joblib.load(pass1_path)
    track_embeddings = data['track_embeddings']
    track_crops      = data['track_crops']
    game_clf         = TeamClassifier()   # fresh instance ready for Pass 2

    sorted_tids = sorted(track_embeddings, key=lambda t: len(track_embeddings[t]), reverse=True)
    sample_tids = [t for t in sorted_tids if t in track_crops][:N_LABEL_CROPS]
    n_clean     = sum(1 for t in sample_tids if track_crops[t][2])
    print(f'Loaded {GAME_SLUG}: {len(track_embeddings)} tracks → '
          f'top {len(sample_tids)} for labeling ({n_clean} clean, '
          f'{len(sample_tids) - n_clean} with player overlap)')

In [ ]:
# Labeling widget — click A / B / X for each crop, then run Pass 2
# Crops with player overlap get a yellow border as a heads-up — label as ? if ambiguous.

COLS = 6
label_toggles = {}
summary_html  = widgets.HTML()

def update_summary(*args):
    counts = {-1: 0, 0: 0, 1: 0, 2: 0}
    for t in label_toggles.values():
        counts[t.value] += 1
    summary_html.value = (
        f"<b style='color:#3498db'>Team A: {counts[0]}</b> &nbsp;|&nbsp; "
        f"<b style='color:#f39c12'>Team B: {counts[1]}</b> &nbsp;|&nbsp; "
        f"<b style='color:#888'>Other: {counts[2]}</b> &nbsp;|&nbsp; "
        f"Unlabeled: {counts[-1]}"
    )

grid_items = []
for tid in sample_tids:
    crop, area, is_clean = track_crops[tid]
    pil  = PILImage.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)).resize((64, 96), PILImage.LANCZOS)
    buf  = io.BytesIO()
    pil.save(buf, format='PNG')

    border_color = '#ddd' if is_clean else '#f39c12'   # yellow border = overlapping players
    img_w   = widgets.Image(value=buf.getvalue(), format='png',
                            layout=widgets.Layout(width='64px', height='96px'))
    toggle  = widgets.ToggleButtons(
        options=[('?', -1), ('A', 0), ('B', 1), ('X', 2)],
        value=-1,
        style={'button_width': '24px', 'font_weight': 'bold'},
    )
    toggle.observe(update_summary, names='value')
    label_toggles[tid] = toggle

    n_embs    = len(track_embeddings[tid])
    tid_label = widgets.HTML(f'<small>#{tid} ({n_embs}x)</small>',
                             layout=widgets.Layout(text_align='center'))
    grid_items.append(widgets.VBox(
        [img_w, toggle, tid_label],
        layout=widgets.Layout(align_items='center', margin='2px', padding='4px',
                              border=f'1px solid {border_color}'),
    ))

rows = [widgets.HBox(grid_items[i:i+COLS]) for i in range(0, len(grid_items), COLS)]
update_summary()
print('Label each crop: A = Team A, B = Team B, X = Other (ref/GK), ? = skip')
print('Yellow border = another player was overlapping in this frame — skip if ambiguous.')
display(summary_html)
display(widgets.VBox(rows))

In [ ]:
# Pass 2: fit supervised classifier → classify all tracks → save
# Run after labeling crops in the widget above.

labeled_embs, labeled_labels = [], []
for tid, toggle in label_toggles.items():
    if toggle.value == -1:
        continue
    for emb in track_embeddings[tid]:
        labeled_embs.append(emb)
        labeled_labels.append(toggle.value)

n_labeled = sum(1 for t in label_toggles.values() if t.value != -1)
print(f'Labeled: {n_labeled} tracks → {len(labeled_embs)} embeddings')

if len(labeled_embs) < 4:
    print('Need at least 4 labeled crops. Go back and label more.')
else:
    labeled_embs_arr   = np.array(labeled_embs,   dtype=np.float32)
    labeled_labels_arr = np.array(labeled_labels,  dtype=np.int8)

    game_clf.fit_supervised(labeled_embs_arr, labeled_labels_arr)

    track_teams = game_clf.classify_tracks(track_embeddings)
    class_names = {0: 'Team A', 1: 'Team B', 2: 'Other'}
    counts = {}
    for t in track_teams.values():
        counts[t] = counts.get(t, 0) + 1
    summary = ', '.join(f'{class_names.get(k, "?")}={v}' for k, v in sorted(counts.items()))
    print(f'Tracks classified: {len(track_teams)} ({summary})')

    # Save fitted model
    clf_path = Config.OUTPUT_CLASSIFIERS_DIR / f'{GAME_SLUG}_classifier.pkl'
    game_clf.save(clf_path)

    # Save raw labeled embeddings + track-level labels (used by review widget)
    # - embeddings / labels: embedding-level (non-skipped), used by refit_knn
    # - review_track_ids / review_track_labels: one entry per track including skipped=-1
    labels_path  = Config.OUTPUT_CLASSIFIERS_DIR / f'{GAME_SLUG}_labels.npz'
    review_tids  = np.array([tid for tid in label_toggles], dtype=np.int64)
    review_lbls  = np.array([label_toggles[tid].value for tid in label_toggles], dtype=np.int8)
    np.savez(labels_path,
        embeddings          = labeled_embs_arr,
        labels              = labeled_labels_arr,
        review_track_ids    = review_tids,
        review_track_labels = review_lbls,
    )
    print(f'Labels saved:     {labels_path.name}  ({len(labeled_embs_arr)} embeddings, {len(review_tids)} tracks)')

    labeled_games = [s for s in Config.MATCH_VIDEOS
                     if (Config.OUTPUT_CLASSIFIERS_DIR / f'{s}_classifier.pkl').exists()]
    print(f'\nProgress: {len(labeled_games)}/{len(Config.MATCH_VIDEOS)} games labeled')
    print('Labeled:', ', '.join(labeled_games))

In [ ]:
# Review widget — crops grouped by assigned label for quality check.
# Works cross-session: loads pass1.pkl + labels.npz from disk.
# After correcting any toggles, click "Save corrections" to update labels.npz and refit.
#
# NOTE: labels.npz must have been saved with the updated Pass 2 cell (above) to include
# review_track_ids / review_track_labels. Games labeled before that update need a Pass 2 re-run.

_REVIEW_SLUG = GAME_SLUG   # change this to review a different game
REVIEW_COLS  = 8

rv_toggles    = {}   # tid → ToggleButton; populated below, used by Save button
_pass1_path   = Config.OUTPUT_CLASSIFIERS_DIR / f'{_REVIEW_SLUG}_pass1.pkl'
_labels_path  = Config.OUTPUT_CLASSIFIERS_DIR / f'{_REVIEW_SLUG}_labels.npz'

if not _pass1_path.exists() or not _labels_path.exists():
    print(f'Missing files for {_REVIEW_SLUG}. Run Pass 1 + Pass 2 first.')
else:
    _pass1        = joblib.load(_pass1_path)
    _review_crops = _pass1['track_crops']
    _review_embs  = _pass1['track_embeddings']
    _npz          = np.load(_labels_path)

    if 'review_track_ids' not in _npz:
        print(f'No review data in {_labels_path.name}.')
        print('Re-run Pass 2 (with the updated cell above) to generate it.')
    else:
        _rv_tids   = _npz['review_track_ids'].tolist()
        _rv_lbls   = _npz['review_track_labels'].tolist()
        _label_map = dict(zip(_rv_tids, _rv_lbls))   # tid → label

        _groups = {0: [], 1: [], 2: [], -1: []}
        for tid, lbl in _label_map.items():
            if tid in _review_crops:
                _groups[lbl].append(tid)

        _GROUP_META = {
             0: ('Team A',  '#3498db'),
             1: ('Team B',  '#f39c12'),
             2: ('Other',   '#888'),
            -1: ('Skipped', '#555'),
        }

        _all_rows = []
        for grp in [0, 1, 2, -1]:
            tids = _groups[grp]
            if not tids:
                continue
            name, color = _GROUP_META[grp]
            _all_rows.append(widgets.HTML(
                f"<h3 style='color:{color};margin:8px 0 4px'>{name} — {len(tids)} crops</h3>"
            ))
            items = []
            for tid in tids:
                crop, area, is_clean = _review_crops[tid]
                pil = PILImage.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)).resize((64, 96), PILImage.LANCZOS)
                buf = io.BytesIO(); pil.save(buf, format='PNG')
                img_w  = widgets.Image(value=buf.getvalue(), format='png',
                                       layout=widgets.Layout(width='64px', height='96px'))
                toggle = widgets.ToggleButtons(
                    options=[('?', -1), ('A', 0), ('B', 1), ('X', 2)],
                    value=grp,
                    style={'button_width': '24px', 'font_weight': 'bold'},
                )
                rv_toggles[tid] = toggle
                n_embs = len(_review_embs.get(tid, []))
                lbl_w  = widgets.HTML(f'<small>#{tid} ({n_embs}x)</small>',
                                      layout=widgets.Layout(text_align='center'))
                border = '#ddd' if is_clean else '#f39c12'
                items.append(widgets.VBox(
                    [img_w, toggle, lbl_w],
                    layout=widgets.Layout(align_items='center', margin='2px', padding='4px',
                                          border=f'1px solid {border}'),
                ))
            for i in range(0, len(items), REVIEW_COLS):
                _all_rows.append(widgets.HBox(items[i:i + REVIEW_COLS]))

        save_btn    = widgets.Button(description='Save corrections', button_style='success',
                                     layout=widgets.Layout(margin='12px 0'))
        save_status = widgets.HTML()

        def _on_save(_):
            rv_embs, rv_flat_lbls, rv_tids_out, rv_lbl_vals = [], [], [], []
            for tid, toggle in rv_toggles.items():
                lbl = toggle.value
                rv_tids_out.append(tid)
                rv_lbl_vals.append(lbl)
                if lbl != -1:
                    for emb in _review_embs.get(tid, []):
                        rv_embs.append(emb)
                        rv_flat_lbls.append(lbl)
            np.savez(
                _labels_path,
                embeddings          = np.array(rv_embs,      dtype=np.float32),
                labels              = np.array(rv_flat_lbls, dtype=np.int8),
                review_track_ids    = np.array(rv_tids_out,  dtype=np.int64),
                review_track_labels = np.array(rv_lbl_vals,  dtype=np.int8),
            )
            clf_path    = Config.OUTPUT_CLASSIFIERS_DIR / f'{_REVIEW_SLUG}_classifier.pkl'
            updated_clf = TeamClassifier.refit_knn(clf_path, _labels_path)
            updated_clf.save(clf_path)
            class_names = {-1: 'Skipped', 0: 'Team A', 1: 'Team B', 2: 'Other'}
            counts = {}
            for v in rv_lbl_vals: counts[v] = counts.get(v, 0) + 1
            summary = '  |  '.join(f"{class_names[k]}: {counts.get(k, 0)}" for k in [0, 1, 2, -1])
            save_status.value = (
                f"<b style='color:green'>Saved {_labels_path.name} "
                f"({len(rv_embs)} embeddings) · classifier refit</b><br>"
                f"<small>{summary}</small>"
            )

        save_btn.on_click(_on_save)

        totals = {k: len(v) for k, v in _groups.items()}
        print(f'Review: {_REVIEW_SLUG}')
        print(f'  Team A: {totals[0]}  |  Team B: {totals[1]}  |  Other: {totals[2]}  |  Skipped: {totals[-1]}')
        print('Adjust any toggles, then click "Save corrections".')
        display(widgets.VBox([save_btn, save_status]))
        display(widgets.VBox(_all_rows))

In [ ]:
# Validation — annotate a 2-minute clip for each harder game + sut-mla (best case baseline)
# Saves to output/classifier_validation/

VALIDATE_SLUGS  = ['mor-bud', 'mor-ars', 'ars-dec', 'mla-bud', 'mla-bud-2', 'pet-bok', 'sut-mla']
VALIDATE_OFFSET = 10    # minutes into first half to start
VALIDATE_DUR    = 120   # seconds per clip (2 min ≈ 3000 frames @ 25fps)
EMBED_EVERY     = 15    # collect embeddings every N frames

import src.visualization
importlib.reload(src.visualization)
from src.visualization import Annotator

_periods  = json.loads((PROJECT_ROOT / 'data' / 'period_detection_results.json').read_text())
_out_dir  = Config.OUTPUT_DIR / 'classifier_validation'
_out_dir.mkdir(parents=True, exist_ok=True)

for VALIDATE_SLUG in VALIDATE_SLUGS:
    print(f'\n{"="*60}\n{VALIDATE_SLUG}\n{"="*60}')

    _clf_path = Config.OUTPUT_CLASSIFIERS_DIR / f'{VALIDATE_SLUG}_classifier.pkl'
    if not _clf_path.exists():
        print(f'  No classifier — skipping.')
        continue

    _period      = next(r for r in _periods if r['slug'] == VALIDATE_SLUG)
    _fh_start    = _period['first_half_start_frame']
    _video_path  = Path(str(Config.MATCH_VIDEOS[VALIDATE_SLUG]))
    _cap         = cv2.VideoCapture(str(_video_path))
    _fps_v       = _cap.get(cv2.CAP_PROP_FPS)
    _start_frame = _fh_start + int(VALIDATE_OFFSET * 60 * _fps_v)
    _n_frames    = int(VALIDATE_DUR * _fps_v)

    _val_clf      = TeamClassifier.load(_clf_path)
    _val_detector = PlayerDetector()
    _annotator    = Annotator()

    # ── Pass 1: detect + track + collect embeddings ───────────────────────
    _track_embs = {}
    _frame_dets = []

    _cap.set(cv2.CAP_PROP_POS_FRAMES, _start_frame)
    for _i in tqdm(range(_n_frames), desc=f'  {VALIDATE_SLUG} pass 1 (detect)'):
        _ret, _frame = _cap.read()
        if not _ret:
            break
        _dets = _val_detector.detect_with_tracking(_frame)
        _frame_dets.append(_dets)
        if _i % EMBED_EVERY == 0 and _dets['players']:
            _embs = _val_clf.collect_embeddings(_frame, _dets['players'])
            for _p, _emb in zip(_dets['players'], _embs):
                _tid = _p.get('track_id', -1)
                if _tid >= 0:
                    _track_embs.setdefault(_tid, []).append(_emb)

    _track_teams = _val_clf.classify_tracks(_track_embs)
    _class_names = {0: 'Team A', 1: 'Team B', 2: 'Other'}
    _counts = {}
    for _t in _track_teams.values():
        _counts[_t] = _counts.get(_t, 0) + 1
    print('  Tracks: ' + ',  '.join(f"{_class_names.get(k, '?')}={v}" for k, v in sorted(_counts.items())))

    # ── Pass 2: re-read frames and write annotated video ──────────────────
    _out_path = _out_dir / f'{VALIDATE_SLUG}_team_check.mp4'
    _h = int(_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    _w = int(_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    _writer = cv2.VideoWriter(str(_out_path), cv2.VideoWriter_fourcc(*'mp4v'), _fps_v, (_w, _h))

    _cap.set(cv2.CAP_PROP_POS_FRAMES, _start_frame)
    for _dets in tqdm(_frame_dets, desc=f'  {VALIDATE_SLUG} pass 2 (annotate)'):
        _ret, _frame = _cap.read()
        if not _ret:
            break
        for _p in _dets['players']:
            _p['team_id'] = _track_teams.get(_p.get('track_id', -1), 0)
        _writer.write(_annotator.annotate_frame(_frame, _dets))

    _writer.release()
    _cap.release()
    print(f'  Saved → {_out_path}')

print('\nAll done.')


sut-mla
TeamClassifier: ResNet18 CNN embeddings (512→32-dim PCA)
Classifier loaded (supervised): C:\Users\PC\Desktop\GitHub\football-computer-vision\output\classifiers\sut-mla_classifier.pkl


  sut-mla pass 1 (detect): 100%|██████████| 3000/3000 [02:31<00:00, 19.78it/s]


  Tracks: Team A=79,  Team B=77,  Other=18


  sut-mla pass 2 (annotate): 100%|██████████| 3000/3000 [00:35<00:00, 84.87it/s]

  Saved → C:\Users\PC\Desktop\GitHub\football-computer-vision\output\classifier_validation\sut-mla_team_check.mp4

All done.
